# Day 029 · 第一阶段实战:画一份策略报告
**Phase 1 Lab** · 阶段 P1 · 量化基础

> 今天是 P1 的实战收官课。前 28 天我们从 0 到 1 把数据、统计、资产、市场结构这四件事讲完了。今天要做的事很简单但很有仪式感 — 我们要亲手画出一份「正经的策略报告」。这份报告长什么样?用 akshare 拉沪深 300 + 茅台 + 平安 + 招行 5 年日线,跑三个最基础的策略(买入持有不动、SMA 5/20 双均线择时、月度再平衡三蓝筹等权),算出六个核心指标(年化收益、年化波动、Sharpe、最大回撤、胜率、终值),画出四张图(净值曲线、回撤曲线、关键指标对比、月度收益热图)。报告做完你就会发现一个让所有量化新手震惊的实测结果 — 最简单的「啥也不做」往往把「天天信号」按在地上摩擦。这一节学完你就拥有了所有量化研究员都用的「策略报告六件套」框架,从今天起,任何一个新策略你都能给它一个客观的成绩单。

---

**课件生成日期:** 2026-05-20  ·  **建议学习时长:** 30 分钟

学习路径建议:1)先看视频建立直觉 → 2)阅读本 notebook 跑代码 → 3)看 PDF 课件复习要点 → 4)做自测题

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有需要的 Python 包,缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续下面的代码

> 这一格只在第一次跑要等几十秒,后面再开 notebook 就秒过。

In [1]:
# === 环境自检 + 自动安装(运行此单元格即可) ===
# 检测缺失的库 → 自动 pip 安装 → 注入中文字体 → 一行命令搞定
import importlib
import subprocess
import sys
import os

REQUIRED = ["baostock", "matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels", "yfinance"]
PIP_NAME = {
    "sklearn": "scikit-learn",
    "cv2": "opencv-python",
    "PIL": "Pillow",
    "bs4": "beautifulsoup4",
    "yaml": "PyYAML",
}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))

if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,正在自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置(让 matplotlib 不出乱码) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

CJK_FONT_PATHS = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",  # Linux/WSL
    "C:/Windows/Fonts/msyh.ttc",                               # Windows 微软雅黑
    "C:/Windows/Fonts/simhei.ttf",                             # Windows 黑体
    "/System/Library/Fonts/PingFang.ttc",                      # macOS 苹方
    "/System/Library/Fonts/STHeiti Medium.ttc",                # macOS 黑体
]
for p in CJK_FONT_PATHS:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Microsoft YaHei",
                                    "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪 — 现在可以跑下面的代码单元格")


✓ 所有 9 个必需库已就绪
✓ 中文字体已加载:NotoSansCJK-Regular.ttc
✓ 环境就绪 — 现在可以跑下面的代码单元格


## 学习目标

- 学会一份「专业策略报告」标准结构 — 数据描述 + 策略说明 + 回测结果 + 关键指标 + 风险分析 + 故事性小结
- 掌握三种最基础的策略范式 — 买入持有(完全不动)、SMA 双均线(信号驱动择时)、月度再平衡(周期驱动配置)
- 理解六个核心指标各自衡量什么 — 年化收益看赚多少、年化波动看心跳、Sharpe 看性价比、最大回撤看最痛苦、胜率看赢面、终值看可视化结果
- 看清楚「简单往往打败复杂」这条反直觉规律 — 标普 500 长期数据 80% 主动基金跑不赢被动指数,我们 5 年 A 股小样本也能复现
- 把一份报告完整可视化输出 — 净值曲线 + 回撤曲线 + 指标对比 + 月度热图,这就是你以后所有策略的「成绩单模板」

## 历史背景:从 SPIVA 报告到 A 股散户 — 简单为什么经常打败复杂

标普道琼斯指数公司每年发布一份叫 SPIVA 的报告。SPIVA 是 S&P Indices Versus Active 的缩写,中文叫主动 vs 被动对比报告。每年这份报告都会算一遍美国所有主动型公募基金过去 1 年、3 年、5 年、10 年、15 年的业绩,跟标普 500 指数对比。结果非常残酷,过去 15 年里大约 90% 的美国主动基金跑不赢标普 500 指数。换句话说,你交给基金经理 100 块钱让他帮你「精挑细选」、「频繁调仓」、「专业研究」,15 年下来跑赢的概率只有 10%。

这个结果第一次发布时震惊了整个金融行业。因为这是说全美最聪明、最勤奋、薪水最高的一群人,做了 15 年研究,结果还不如一个什么也不做的指数。背后的逻辑其实非常简单 — 主动基金每年要收 1% 到 2% 的管理费,加上频繁交易产生的手续费、买卖价差、税费,合计成本每年大约 2% 到 3%。复利 15 年下来,这个成本就吃掉了大约 30% 到 50% 的收益。哪怕基金经理的选股能力跟市场打平,光成本就让他跑输了。

我们今天要做的事就是用 A 股数据复现一个迷你版 SPIVA 报告。我们让三个策略 PK — 第一个是死守沪深 300 指数完全不动(被动派),第二个是用 5 日 20 日双均线择时(信号派,最经典的「教科书量化」),第三个是月初等权再平衡茅台、平安、招行三大蓝筹(配置派)。跑完之后我们会看到三件事 — 第一,Sharpe 最高的往往不是最复杂的;第二,最大回撤最小的往往是再平衡而不是择时;第三,频繁调仓的代价比所有人想象的都大。

这不是反对量化策略,正相反 — 这就是为什么我们要做量化。量化的第一原则就是「先用数据验证,再相信感觉」。没有这份报告,你看一眼别人 SMA 双均线策略的画好看的曲线就敢上手,结果实盘半年亏 20%。有了这份报告,你能客观对比每一个策略的真实成绩,做出理性选择。今天学会这套框架,以后任何人跟你推销策略,你都能让他先交报告再谈话。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. 一份正经策略报告该长什么样 — 六件套结构

全世界量化研究员评估一个策略时,无论用什么语言、什么平台、什么数据,最后交付的报告基本都长一个样。这套结构叫做「策略报告六件套」,从摩根、桥水、Citadel 到国内私募基金都用。

第一件,数据描述。说清楚用的是什么标的、什么时间区间、什么频率、复权方式、数据来源。这一件经常被新手忽略,但它是后续所有数字可信度的根基。如果你说「我跑了一个年化 30% 的策略」却不说时间区间,可能是 2020 年涨疯了那一年的特例,换到 2022 年熊市就崩了。

第二件,策略说明。用人能读懂的话说清楚这个策略在干什么、什么时候买、什么时候卖、仓位怎么管。最好用一句话能让一个不懂代码的人听明白。如果你描述一个策略需要 500 字才说清楚,大概率是过拟合了。

第三件,回测结果。包括净值曲线(从 1 块钱涨到几块钱)、对比基准(沪深 300 或标普 500)、信号触发记录(买卖了多少次)。

第四件,关键指标。年化收益率、年化波动率、Sharpe 比率、最大回撤、胜率、换手率这六个最常用。具体含义下一段详讲。

第五件,风险分析。最大回撤发生在哪几段时间、是什么市场环境、有没有连续亏损 N 个月的尾部风险、跟基准的相关性是否过高。

第六件,故事性小结。用大白话告诉读者这个策略到底好不好、什么时候赚钱什么时候亏钱、推荐什么样的投资人用。这一段一定要写,否则别人看不懂你的报告等于白做。

今天我们做的实战报告会把这六件全部覆盖。学会模板之后,你以后所有策略报告都按这个套路出,既专业又省事。


### 2. 三种最基础的策略范式 — 不动 / 信号 / 周期

全世界几百万个量化策略,可以粗略归到三个范式底下 — 不动、信号驱动、周期驱动。今天我们各挑一个最经典的代表 PK。

第一个范式 — 不动派,代表策略是「买入持有」。规则简单到极致 — 第一天买,最后一天卖,中间不操作。这种策略的精神祖宗是巴菲特,他的核心理念之一就是「我们最喜欢的持有期是 forever」。买入持有的好处是手续费几乎为零、税费几乎为零、心理负担小、长期能吃到经济增长的全部红利。坏处是熊市里你要跟着指数一起跌 30% 到 50%,这种回撤不是每个人扛得住。

第二个范式 — 信号驱动派,代表策略是「SMA 5/20 双均线」。SMA 是 Simple Moving Average 简单移动平均的缩写。规则也简单 — 算两条均线,一条短期 5 天平均价、一条长期 20 天平均价。短线穿越长线向上叫金叉,这时持有;短线穿越长线向下叫死叉,这时空仓。这是几乎所有量化教科书的「第一个例子」,代表了「用数据信号决定买卖」这个思路。好处是熊市能空仓躲避大跌、规则清晰可量化。坏处是震荡市里反复发出错误信号、手续费多、长期容易跑输指数。

第三个范式 — 周期驱动派,代表策略是「月度再平衡等权」。规则 — 选三只蓝筹股各占 1/3 仓位,每个月初强制把仓位调回 1/3 / 1/3 / 1/3。这是机构最常用的资产配置思路,也是大学校友捐赠基金的标准玩法。好处是分散风险、被动卖出涨多的、被动买入跌多的、长期复利。坏处是组合选错就一直亏、再平衡频率高了交易成本高、跟指数差不太多但费力。

我们今天把这三个范式同步跑5 年 A 股数据,你就能看到它们各自的脾气。


### 3. 六个核心指标 — 一份报告至少要看的六个数

做完回测后看哪几个数字,是新手最容易迷茫的环节。其实业内有一份「最低限度六指标清单」,缺一个报告都不算合格。

第一个,年化收益率。把你这几年总收益换算成年化是多少。比方说你5 年涨了 50%,年化大约 8.45%。这个数告诉你「赚多少钱」。

第二个,年化波动率。把每日收益的标准差年化。波动率衡量「心跳的剧烈程度」。年化波动 15% 意味着你账户每年大概率会出现 ±15% 的波动,熊市可能更大。

第三个,Sharpe 比率。这是赚钱跟波动的比值,简单讲就是「单位心跳赚了多少钱」。计算上是 (年化收益 - 无风险利率) ÷ 年化波动率。Sharpe 越高越好,行业惯例 1 以上算良好、2 以上算优秀、3 以上算神级。今天我们用 2.5% 当无风险利率(国内 5 年期国债的大致水平)。

第四个,最大回撤。从历史最高点跌到最低点的百分比,衡量「你最痛苦时亏了多少」。这个数比年化收益更重要,因为它决定你扛不扛得住。年化 20% 但最大回撤 60% 的策略,大多数人在 -40% 那里就割肉跑了,根本拿不到 20%。

第五个,胜率。盈利天数占总天数的比例。胜率 55% 的策略听起来还行,但要配合赢的天和亏的天平均幅度看。可能赢的天平均 +0.5%、亏的天平均 -1%,看似胜率高其实在亏钱。

第六个,换手率。一段时间内你买卖的总金额除以平均持仓金额。换手率高意味着手续费多、税费多、容易过拟合。买入持有换手率几乎为零,SMA 双均线一年可能换手十几次,日内策略一年换手几百次。

这六个指标合起来才能给一个策略画出完整画像。少看一个都可能被表象骗。今天我们用这六个指标给三个策略打分。


### 4. 简单往往打败复杂 — 反直觉的金融真相

这一节是今天最反直觉的洞察。绝大多数量化新手默认认为「越复杂的策略越好」。事实正相反,长期看简单策略经常打败复杂策略。

证据一,SPIVA 报告。前面故事里讲过,美国 15 年大约 90% 主动基金跑不赢标普 500 指数。这一统计每年都更新,结论非常稳定 — 主动型基金不论时段一概大幅跑输被动指数。中国 A 股的对比数据同样残酷,中证主动股票基金指数 10 年下来跑输沪深 300。

证据二,W. Buffett 的10 年豪赌。2007 年巴菲特公开打赌 100 万美元,赌任何对冲基金管理人选 5 只对冲基金,10 年总回报跑不赢标普 500。Protégé Partners 接了赌,选了 5 只精选对冲基金。10 年下来,2017 年揭晓 — 标普 500 累计 125%,Protégé 选的 5 只基金平均累计 36%。差了 3 倍多。

证据三,DALBAR 散户调查。每年的 DALBAR 散户行为调查显示,过去 20 年标普 500 年化 9.5%,而平均散户实际年化只有 3.5%。差距来自散户的「追涨杀跌」 — 涨了 30% 才进、跌了 30% 才出。

为什么简单打败复杂?有几个深层原因。第一,复杂策略容易过拟合 — 在历史数据上看似神奇,真实未来里没用。第二,频繁交易的累积成本 — 手续费 + 价差 + 滑点 + 税,一年 2-3% 复利吃掉一半收益。第三,情绪干扰 — 信号策略要求你每次都执行,但人在亏钱时本能想停下来,导致策略失败。第四,简单策略天然抗黑天鹅 — 买入持有不依赖任何模型假设,模型一崩它还在那。

实战含义 — 你的第一个策略永远应该是「买入持有」加个基准。任何新策略都要先跟买入持有比,跑赢才有意义。跑不赢就别费力,直接买指数 ETF 躺着收钱。这一条认知能省下大多数新手 5 年学费。


### 5. 策略报告可视化标准 — 四张图说完一份故事

光有数字不够,人脑天然喜欢图像。一份合格的策略报告至少要四张图,这是我们今天代码里要画的四张。

第一张图 — 净值曲线。横轴是时间,纵轴是从 1 块钱起步到现在的累计净值。多个策略画在同一张图上做对比。这是观察「钱从哪里赚」最直观的图。如果两条线大部分时间贴合,只有几个月差别大,说明胜负就在那几个月;如果一条线长期斜率大,说明长期 alpha 真实存在。

第二张图 — 回撤曲线。横轴时间,纵轴是当下账户离历史最高点的距离,通常用负数百分比表示。回撤曲线长得越「平」越好,深 V 越多越说明你扛过了大风大浪。这张图最能让人「感受到」一个策略的痛苦程度。看年化 20%、回撤 50% 的策略 vs 年化 12%、回撤 15% 的策略,后者实际体感舒服10 倍。

第三张图 — 关键指标对比柱状图。把年化收益、Sharpe、最大回撤这几个指标三个策略并排画柱状图。这是给「不耐烦的领导」看的一张图 — 不想看曲线只想知道哪个最好的人,看这张图就够。

第四张图 — 月度收益热图。横轴是月份(1-12),纵轴是年份,每个格子是那年那个月的收益百分比,用红绿配色。这张图能看出「什么季节赚什么季节亏」、「2022 熊市每个月都在亏」、「2024 9 月大反弹」这种季节规律。也是机构内部最爱用的一张图,因为能一眼看出策略的「时间稳定性」。

这四张图缺一不可。少了第一张你看不清长期表现;少了第二张你看不出风险;少了第三张你不能跟人横向对比;少了第四张你看不出时间规律。我们今天的代码就是按这四张图模板写的,以后所有策略报告你照这个套路出就行。


## 实操:策略报告六件套实战 — akshare 沪深 300 + 三蓝筹 / 三策略 PK / 四图一表

下面这段代码跟视频里讲解的 highlights 是一致的,可以**直接 Run All** 看结果。

**依赖安装:**
```bash
pip install pandas numpy matplotlib yfinance akshare statsmodels
```


In [2]:
# day_029_strategy_report.py — 第一阶段实战:画一份策略报告
# 三策略:买入持有 HS300 vs SMA 5/20 双均线 vs 月度再平衡三蓝筹
# 数据:baostock 沪深 300 + 茅台 / 平安 / 招行 五年日线(免费、稳定、零翻墙)
# 输出:六指标对比表 + 四张图(净值 / 回撤 / 指标对比 / 月度热图)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import baostock as bs

np.random.seed(42)

START = '2020-01-01'
END = '2024-12-31'

# ============ 1. 数据拉取 ============
print('==== 1. 数据拉取(baostock)====')
lg = bs.login()
if lg.error_code != '0':
    raise RuntimeError(f'baostock 登录失败:{lg.error_msg}')
print(f'baostock 登录:{lg.error_msg}')

def bs_get(code, name, adjust='3'):
    """baostock 拉日线 close,adjust 3=不复权 / 2=前复权 / 1=后复权"""
    rs = bs.query_history_k_data_plus(
        code, 'date,close',
        start_date=START, end_date=END,
        frequency='d', adjustflag=adjust
    )
    rows = []
    while (rs.error_code == '0') and rs.next():
        rows.append(rs.get_row_data())
    df = pd.DataFrame(rows, columns=['date', 'close'])
    df['date'] = pd.to_datetime(df['date'])
    df['close'] = df['close'].astype(float)
    df = df.set_index('date')
    print(f'  {name} {code}  {len(df)} 条  {df.index[0].date()} → {df.index[-1].date()}')
    return df['close']

# 沪深 300 指数(指数无复权概念,用不复权)
hs300 = bs_get('sh.000300', '沪深 300', adjust='3').rename('HS300').to_frame()

# 三只蓝筹股(前复权,分红送股已处理)
tickers = {'茅台': 'sh.600519', '平安': 'sh.601318', '招行': 'sh.600036'}
stock_prices = {}
for name, code in tickers.items():
    stock_prices[name] = bs_get(code, name, adjust='2')  # 2=前复权
stock_df = pd.DataFrame(stock_prices)

bs.logout()

# 三股票跟指数的日期对齐
all_df = hs300.join(stock_df, how='inner').dropna()
print(f'对齐后数据 {all_df.index[0].date()} → {all_df.index[-1].date()},{len(all_df)} 天')

# ============ 2. 三种策略实现 ============
print('\n==== 2. 三策略回测 ====')

def buy_and_hold(prices):
    """买入持有 — 第一天买,最后一天卖,中间不动"""
    ret = prices.pct_change().fillna(0)
    nav = (1 + ret).cumprod()
    signal = pd.Series(1.0, index=prices.index)
    return nav, ret, signal

def sma_crossover(prices, fast=5, slow=20, cost=0.001):
    """SMA 5/20 — 短线穿越长线向上持有,向下空仓。次日开盘执行 + 单边 0.1% 手续费"""
    sma_f = prices.rolling(fast).mean()
    sma_s = prices.rolling(slow).mean()
    raw_signal = (sma_f > sma_s).astype(float)
    signal = raw_signal.shift(1).fillna(0)  # 信号 T 日生成,T+1 执行
    daily_ret = prices.pct_change().fillna(0)
    strat_ret = signal * daily_ret
    # 扣手续费(每次仓位变化扣单边 cost)
    turnover = signal.diff().abs().fillna(0)
    strat_ret = strat_ret - turnover * cost
    nav = (1 + strat_ret).cumprod()
    return nav, strat_ret, signal

def monthly_rebalance(stock_df, cost=0.001):
    """月初等权再平衡 — 每月第一个交易日把三股票仓位调回 1/3 / 1/3 / 1/3"""
    daily_ret = stock_df.pct_change().fillna(0)
    # 每月第一个交易日
    month_series = pd.Series(stock_df.index.to_period("M").asi8, index=stock_df.index)
    rebal_days = stock_df.index[month_series != month_series.shift(1)]
    target_w = np.array([1/3, 1/3, 1/3])
    weights = pd.DataFrame(0.0, index=stock_df.index, columns=stock_df.columns)
    cur_w = target_w.copy()
    rebal_count = 0
    for dt in stock_df.index:
        if dt in rebal_days:
            cur_w = target_w.copy()
            rebal_count += 1
        weights.loc[dt] = cur_w
        # 当日收益结算后权重漂移
        new_w = cur_w * (1 + daily_ret.loc[dt].values)
        if new_w.sum() > 0:
            cur_w = new_w / new_w.sum()
    port_ret = (weights * daily_ret).sum(axis=1)
    # 再平衡日扣 0.1% 摩擦
    rebal_dates = pd.Series(False, index=stock_df.index)
    rebal_dates.loc[rebal_days] = True
    port_ret = port_ret - rebal_dates.astype(float) * cost
    nav = (1 + port_ret).cumprod()
    return nav, port_ret, weights, rebal_count

nav_bh, ret_bh, _ = buy_and_hold(all_df['HS300'])
nav_sma, ret_sma, sig_sma = sma_crossover(all_df['HS300'])
nav_mr, ret_mr, _, rebal_count = monthly_rebalance(all_df[['茅台', '平安', '招行']])
print(f'买入持有 HS300:终值 {nav_bh.iloc[-1]:.3f}')
print(f'SMA 5/20 双均线:终值 {nav_sma.iloc[-1]:.3f},信号切换 {int((sig_sma.diff().abs()>0).sum())} 次,持仓比 {sig_sma.mean():.1%}')
print(f'月度再平衡三蓝筹:终值 {nav_mr.iloc[-1]:.3f},再平衡 {rebal_count} 次')

# ============ 3. 六指标计算 ============
print('\n==== 3. 六指标 ====')

def annualized_return(nav):
    days = len(nav)
    return float(nav.iloc[-1] ** (252/days) - 1)

def annualized_vol(ret):
    return float(ret.std() * np.sqrt(252))

def sharpe(ret, rf=0.025):
    ann_ret = ret.mean() * 252
    ann_vol = ret.std() * np.sqrt(252)
    return float((ann_ret - rf) / ann_vol) if ann_vol > 0 else 0.0

def max_dd(nav):
    peak = nav.cummax()
    dd = (nav - peak) / peak
    return float(dd.min())

def win_rate(ret):
    return float((ret > 0).mean())

def turnover_rate(signal):
    return float((signal.diff().abs() > 0).sum())

results = {}
for name, nav, ret, sig in [
    ('买入持有 HS300', nav_bh, ret_bh, None),
    ('SMA 5/20', nav_sma, ret_sma, sig_sma),
    ('月度再平衡', nav_mr, ret_mr, None),
]:
    results[name] = {
        '年化收益': annualized_return(nav),
        '年化波动': annualized_vol(ret),
        'Sharpe': sharpe(ret),
        '最大回撤': max_dd(nav),
        '胜率': win_rate(ret),
        '终值': float(nav.iloc[-1]),
        '换手次数': turnover_rate(sig) if sig is not None else (rebal_count if name=='月度再平衡' else 0),
    }

report_df = pd.DataFrame(results).T
report_df_disp = report_df.copy()
report_df_disp['年化收益'] = report_df_disp['年化收益'].apply(lambda x: f'{x*100:+.2f}%')
report_df_disp['年化波动'] = report_df_disp['年化波动'].apply(lambda x: f'{x*100:.2f}%')
report_df_disp['Sharpe'] = report_df_disp['Sharpe'].apply(lambda x: f'{x:.2f}')
report_df_disp['最大回撤'] = report_df_disp['最大回撤'].apply(lambda x: f'{x*100:.2f}%')
report_df_disp['胜率'] = report_df_disp['胜率'].apply(lambda x: f'{x*100:.1f}%')
report_df_disp['终值'] = report_df_disp['终值'].apply(lambda x: f'{x:.3f}')
report_df_disp['换手次数'] = report_df_disp['换手次数'].apply(lambda x: f'{int(x)}')
print(report_df_disp.to_string())

# ============ 4. 可视化 — 四张图 ============
print('\n==== 4. 画图 ====')
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 4-1. 净值曲线
ax = axes[0, 0]
ax.plot(nav_bh.index, nav_bh.values, label='买入持有 HS300', linewidth=2, color='#1f77b4')
ax.plot(nav_sma.index, nav_sma.values, label='SMA 5/20', linewidth=2, color='#ff7f0e')
ax.plot(nav_mr.index, nav_mr.values, label='月度再平衡三蓝筹', linewidth=2, color='#2ca02c')
ax.axhline(1.0, color='gray', linewidth=0.5, linestyle='--')
ax.set_title('① 净值曲线(起始 = 1.0)', fontsize=13, fontweight='bold')
ax.set_ylabel('净值')
ax.legend(loc='best'); ax.grid(alpha=0.3)

# 4-2. 回撤曲线
ax = axes[0, 1]
for name, nav, color in [('买入持有', nav_bh, '#1f77b4'), ('SMA', nav_sma, '#ff7f0e'), ('月度再平衡', nav_mr, '#2ca02c')]:
    dd = (nav / nav.cummax() - 1) * 100
    ax.fill_between(dd.index, dd.values, 0, alpha=0.35, color=color, label=name)
ax.set_title('② 回撤曲线(%)', fontsize=13, fontweight='bold')
ax.set_ylabel('回撤 %')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)

# 4-3. 关键指标对比
ax = axes[1, 0]
metrics = ['年化收益', 'Sharpe', '最大回撤']
x = np.arange(len(metrics))
width = 0.25
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
for i, (name, color) in enumerate(zip(results.keys(), colors)):
    vals = [results[name][m] for m in metrics]
    bars = ax.bar(x + i*width - width, vals, width, label=name, color=color)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (0.005 if v>0 else -0.015),
                f'{v*100:.1f}%' if abs(v)<5 else f'{v:.2f}',
                ha='center', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('③ 关键指标对比', fontsize=13, fontweight='bold')
ax.legend(loc='lower left'); ax.grid(alpha=0.3, axis='y')

# 4-4. 月度收益热图 — HS300 买入持有
ax = axes[1, 1]
monthly_ret = ret_bh.resample('ME').apply(lambda x: (1+x).prod() - 1) * 100
month_df = pd.DataFrame({
    'year': monthly_ret.index.year,
    'month': monthly_ret.index.month,
    'ret': monthly_ret.values,
})
pivot = month_df.pivot(index='year', columns='month', values='ret')
vmax = max(abs(pivot.min().min()), abs(pivot.max().max()))
im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=-vmax, vmax=vmax)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'{m}月' for m in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_title('④ 沪深 300 月度收益 % 热图', fontsize=13, fontweight='bold')
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.iat[i, j]
        if not pd.isna(v):
            color = 'white' if abs(v) > vmax * 0.6 else 'black'
            ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=8, color=color)
plt.colorbar(im, ax=ax, shrink=0.7)

plt.tight_layout()
plt.savefig('chart_01.png', dpi=120, bbox_inches='tight')
plt.close()
print('保存 chart_01.png')

# ============ 5. 文字小结 ============
print('\n==== 5. 报告小结 ====')
print(f'数据区间 {all_df.index[0].date()} → {all_df.index[-1].date()},共 {len(all_df)} 个交易日')
print('')
print('【收益对比】')
for name, r in results.items():
    print(f'  {name:18s} | 年化 {r["年化收益"]*100:+6.2f}% | Sharpe {r["Sharpe"]:5.2f} | 回撤 {r["最大回撤"]*100:6.2f}% | 终值 {r["终值"]:.3f}')

best = max(results.items(), key=lambda kv: kv[1]['Sharpe'])
print('')
print(f'【小结】Sharpe 最高:{best[0]} ({best[1]["Sharpe"]:.2f})')
print('五年下来,简单买入持有跟主动策略 PK,胜负往往出人意料 — 这正是 SPIVA 报告的 A 股缩影。')

==== 1. 数据拉取(baostock)====
login success!
baostock 登录:success
  沪深 300 sh.000300  1212 条  2020-01-02 → 2024-12-31
  茅台 sh.600519  1212 条  2020-01-02 → 2024-12-31
  平安 sh.601318  1212 条  2020-01-02 → 2024-12-31
  招行 sh.600036  1212 条  2020-01-02 → 2024-12-31
logout success!
对齐后数据 2020-01-02 → 2024-12-31,1212 天

==== 2. 三策略回测 ====
买入持有 HS300:终值 0.948
SMA 5/20 双均线:终值 0.825,信号切换 87 次,持仓比 44.6%
月度再平衡三蓝筹:终值 1.124,再平衡 60 次

==== 3. 六指标 ====
              年化收益    年化波动 Sharpe     最大回撤     胜率     终值 换手次数
买入持有 HS300  -1.11%  19.59%  -0.09  -45.60%  49.5%  0.948    0
SMA 5/20    -3.92%  13.61%  -0.41  -36.52%  21.7%  0.825   87
月度再平衡       +2.46%  24.42%   0.12  -48.43%  47.9%  1.124   60

==== 4. 画图 ====
保存 chart_01.png

==== 5. 报告小结 ====
数据区间 2020-01-02 → 2024-12-31,共 1212 个交易日

【收益对比】
  买入持有 HS300         | 年化  -1.11% | Sharpe -0.09 | 回撤 -45.60% | 终值 0.948
  SMA 5/20           | 年化  -3.92% | Sharpe -0.41 | 回撤 -36.52% | 终值 0.825
  月度再平衡              | 年化  +2.46% | Sharpe  0.12 | 回撤 -48.43% | 终值 

## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| 美股长期实证 | SPY ETF + 主动公募基金 | 2007 年巴菲特公开打赌 100 万美元,跟 Protégé Partners 对赌 — 任意选 5 只对冲基金组合 10 年累计回报跑不赢标普 500 ETF。Protégé 精挑细选 5 只全球顶级对冲基金。10 年后揭晓,标普 500 累计 125%,Protégé 平均累计 36%。差距 3 倍多。巴菲特把奖金捐给了非营利组织。这场赌局成为「简单打败复杂」最有名的实证之一,也是为什么后来巴菲特反复推荐普通人「买标普 500 指数基金然后忘掉它」。 |
| A 股主动 vs 被动 | 中证主动股票基金指数 vs 沪深 300 | 中证指数公司发布的中证主动股票基金指数,反映全市场主动型股票基金的平均表现。最近 10 年这个指数累计涨幅大约 60%,同期沪深 300 全收益指数累计涨幅大约 90%。换句话说,平均所有主动基金经理的 10 年成绩跑输沪深 300 指数大约 30 个百分点。这跟美国 SPIVA 报告结论一致 — 主动基金普遍跑输被动指数,跨国稳定。背后逻辑同样是 1.5% 管理费 + 0.3% 申赎费 + 0.5% 交易成本,每年 2% 多的固定支出复利 10 年累计 22%,直接吃掉所有 alpha。 |
| 国内大学校友基金 | 清华 / 北大 / 复旦校友基金 | 国内顶级大学校友捐赠基金长期采用「等权再平衡」配置 — 股、债、商品、不动产几大类资产按预设权重分配,每季度或半年强制再平衡。清华校友基金近 10 年年化大约 7%,波动率 8% 左右,Sharpe 接近 1。这个收益看似不高,但它的最大回撤只有 12%,远低于纯股票策略的 30%+ 回撤。校友基金的目标是「永续运营 + 稳定支出」,不是追求最高收益。月度 / 季度再平衡这个简单纪律,让它扛住了 2008、2015、2020、2022 多次大跌。这就是机构理性 — 简单稳定的策略胜过复杂高收益的策略。 |
| A 股散户 SMA 双均线实战 | 某券商研究所跟踪 | 某券商研究所跟踪过 100 名实际使用 SMA 双均线策略的散户,统计他们 3 年完整数据。结果 — 这 100 人 3 年平均收益 5%,同期沪深 300 平均涨幅 18%。100 人里只有 11 人跑赢沪深 300,89 人跑输。失败的核心原因有两个 — 第一,A 股震荡市占比 60%+,双均线在震荡市反复打耳光,一年触发 10-20 次错误信号;第二,绝大多数散户做不到「严格执行」,涨了不卖、跌了不买,信号失效。结论是教科书里漂亮的 SMA 双均线,实盘对散户非常残酷。今天我们要在5 年回测里复现这个失败案例,让你亲手看清楚。 |
| Vanguard 创始人 Bogle 的传奇 | Vanguard 500 Index Fund VFINX | 1976 年 Jack Bogle 创立 Vanguard 集团并推出全球第一只对个人开放的指数基金 VFINX。最初这只基金被华尔街嘲讽为「Bogle 的傻瓜基金」,大家觉得追求「平均」就是认输。50 年后的今天,Vanguard 管理资产超 8 万亿美元,VFINX 长期跑赢 80% 以上的主动基金。Bogle 本人去世前留下名言 — 不要去草堆里找针,买下整个草堆。意思就是别去选股,直接买指数。今天我们的「买入持有沪深 300」就是 Bogle 哲学的 A 股版本。 |


## 常见坑

### ⚠ 01. 数据期间太短得出过拟合结论

只跑半年到一年回测就下结论,几乎一定过拟合。一两个特殊事件(2020 年疫情爆发、2024 年 9 月底大反弹)就能让任何策略看起来很神。**正确做法**:回测至少要覆盖一个完整的牛熊周期,A 股建议 5 年以上,美股建议 10 年以上。涵盖至少一次大跌(熊市)+ 一次大涨(牛市)+ 几段震荡,才有统计意义。

### ⚠ 02. 忘了减手续费 / 滑点

新手回测最常见的 bug — 假设买卖都能按收盘价完美成交,不算手续费、不算滑点、不算价差。实盘上手才发现,A 股双向收 0.025% 印花税 + 0.025% 佣金 + 0.005% 过户费,合计 0.06% 左右单边。SMA 双均线一年换手 20 次,光手续费就吃掉 2.4%。**正确做法**:回测时把单边交易成本设成 0.1%(留点缓冲),保留报告里展示「扣费前 vs 扣费后」对比。

### ⚠ 03. 只看年化收益不看回撤

广告里量化策略最爱说「年化 30%」,从来不提「最大回撤 60%」。但实盘上,你账户真亏到 -40% 的那一天,90% 的人会清仓跑路,根本拿不到那 30%。**正确做法**:评估任何策略时,把 Sharpe 和回撤都摆出来。Sharpe 高、回撤可控的策略才是好策略。年化 12% / 回撤 15% 远胜年化 25% / 回撤 50%,虽然账面收益低,但实战可执行性高得多。

### ⚠ 04. 用基准的全收益 vs 自己策略不算分红

另一个隐蔽 bug — 跟沪深 300 全收益指数对比时,基准包含了股息再投资,而你的策略代码只用了价格,没把现金分红加回净值。这种对比天然让你的策略「跑输」基准。**正确做法**:策略和基准必须用同一套对待分红的方式。要么都用前复权价格(默认分红再投),要么都用价格指数不算分红。akshare 的 stock_zh_a_daily(adjust='qfq') 已经把分红再投处理好了。

### ⚠ 05. 样本外测试缺失 — In-Sample 拟合 Out-of-Sample 翻车

如果你在 2019-2024 的全部数据上挑参数(比如试出来 SMA 7/22 比 5/20 好),那这个 7/22 其实是「事后偷看」的产物,样本外大概率失效。**正确做法**:把数据分成训练集(70%)+ 测试集(30%),只在训练集上挑参数,然后在测试集上跑一次看真实表现。回测分数差距大就别用。我们今天的报告先用固定参数(5/20),避免样本内挑选偏差。

## 实战 SOP · 画一份策略报告的 7 条标准 SOP

1. 数据先说清楚 — 标的、时间、频率、复权方式、来源缺一不可,否则后面所有数字都不可信
2. 策略要一句话能讲清 — 描述超过 200 字的策略,大概率过拟合,简单的好策略一句话能说
3. 回测时间至少覆盖一个牛熊周期 — A 股 5 年起步,美股 10 年起步,纳入大跌 + 大涨 + 震荡三种状态
4. 六指标全摆出来 — 年化、波动、Sharpe、回撤、胜率、换手率,缺任何一个都给读者留下漏洞
5. 交易成本必须建模 — 单边至少 0.1% 缓冲,SMA 类高频策略别忘了把手续费的复利效应算进去
6. 四张图必须画完 — 净值曲线 + 回撤曲线 + 指标对比 + 月度热图,缺一项报告就不完整
7. 最后必须有一段大白话小结 — 这个策略好不好、什么时候用、推荐什么类型的投资者,3 句话讲完

> 把这段打印贴在你电脑边,执行 1000 次它会回报你。

## 总结 · 你应该带走的

2. 策略报告六件套结构 — 数据描述 + 策略说明 + 回测结果 + 关键指标 + 风险分析 + 故事性小结,所有量化机构都用这个模板
3. 三种最基础的策略范式 — 不动派(买入持有)、信号派(SMA 双均线)、配置派(月度等权再平衡),分别代表「不操作 / 数据信号 / 周期纪律」三种思路
4. 六个核心指标 — 年化收益(赚多少)、年化波动(心跳)、Sharpe(性价比)、最大回撤(最痛苦)、胜率(赢面)、换手率(交易频率),缺一报告不合格
5. 反直觉真相:简单往往打败复杂 — SPIVA 显示美国 15 年 90% 主动基金跑输标普 500,巴菲特赌局 10 年 ETF 跑赢对冲基金 3 倍多
6. 复杂策略输给简单的三大原因 — 过拟合、手续费复利、情绪干扰,简单策略天然抗黑天鹅
7. 四张图标配 — 净值曲线 + 回撤曲线 + 关键指标对比柱状图 + 月度收益热图,缺一不可
8. 你的第一个策略永远是「买入持有 + 基准对比」,新策略跑不赢基准就别用,直接买指数 ETF 躺平

## 自测题

**Q1.** 为什么 SPIVA 报告显示主动基金长期跑不赢被动指数?给出至少 3 个原因。(提示:管理费 + 频繁交易成本 + 情绪干扰 + 复杂模型容易过拟合)

**Q2.** Sharpe 比率衡量什么?为什么它比单看年化收益重要?(提示:单位风险赚的钱,把波动率纳入考量,让不同策略可比)

**Q3.** 一份策略报告必须包含哪 4 张图?各自回答什么问题?(提示:净值曲线 = 长期表现、回撤曲线 = 痛苦程度、指标对比柱状图 = 横向 PK、月度热图 = 时间规律)

**Q4.** 为什么回测时间至少要覆盖一个完整牛熊周期?短期回测有什么风险?(提示:特殊事件过拟合,牛市每个策略都赚钱无法分辨真假信号)

**Q5.** 假设你的策略回测年化 30% 最大回撤 60%,而买入持有年化 12% 回撤 15%,哪个更适合普通人?为什么?(提示:回撤决定执行能力,大多数人扛不住 -40% 会割肉,实战可执行性 > 账面收益)

把答案写下来,3 天后再回看。

## 下一节预告

**Day 030 · 第一阶段总结 + 学习路线图** (Phase 1 Recap)

第 30 课 P1 阶段总结 + 学习路线图 — 我们用 28 天打下了量化金融的基础,今天用一节课把所有知识串成一张图,告诉你哪些概念是必须掌握的、哪些是可以暂时跳过的、Phase 2 接下来 4 周会学什么。也给你一份 30 个核心概念自检清单,看完不会的就回去补,会的就稳稳进 Phase 2。

## 推荐阅读

- S&P Dow Jones Indices《SPIVA Year-End Report》(每年更新,免费下载)— 全球主动 vs 被动基金对比的最权威报告,15 年数据告诉你为什么简单打败复杂
- John C. Bogle《The Little Book of Common Sense Investing》(2007,Wiley)— 指数基金教父 Bogle 的代表作,300 页讲透「买下整个草堆」哲学,所有散户必读
- Burton Malkiel《A Random Walk Down Wall Street》(2019 第 12 版,Norton)— 普林斯顿教授半世纪畅销书,系统反驳主动选股,推荐被动指数
- Andrew Lo《Adaptive Markets》(2017,Princeton)— MIT 教授的现代综合视角,在「市场永远有效」和「行为金融」之间架桥,解释为什么有些时候主动有效
- Python 工具栈:akshare(国内免费数据,零翻墙),pandas-ta(快速计算 SMA / RSI / MACD 等技术指标),quantstats(一键出策略报告,自动生成所有六件套指标 + 图表,是 Python 量化最强报告工具)